# BEIR A/B: pplx-embed-v2-late-0.6b on BEIR (scifact)Notes specific to this notebook are in the markdown cell below. The harness(`beir_ab.py`) is written by the next cell, so this notebook is self-contained.

## What this notebook testsDoes **pplx-embed-v2-late-0.6b** (ColBERT-style late interaction, 340M activeparams) beat plain dense retrieval enough to justify a rerank stage?**Model:** [perplexity-ai/pplx-embed-v2-late-0.6b](https://huggingface.co/perplexity-ai/pplx-embed-v2-late-0.6b),MIT. Scores 128-dim vectors **per token** and ranks with MaxSim(`sum over query tokens of max over doc tokens`). The 9B sibling scoresViDoRe v3 65.2% image / 64.7% markdown and shares this embedding space, soyou can index with the 9B and query with the 0.6B.**Why there is no PLAID index here.** `pylate` 1.6.0 pins`sentence-transformers==5.3.0` and `transformers<=5.3.0`, but this modelrequires `sentence-transformers>=6.0.0` and `transformers>=5.4.0`. Thoseconstraints are mutually exclusive, so this notebook scores MaxSim by bruteforce instead. At SciFact (5.2k docs) and NFCorpus (3.6k docs) that is fast.It would not be at MS MARCO scale — a real late-interaction index is aPLAID-style structure, roughly 128 floats per token of every document.**Cost shape.** Indexing is expensive (per-token vectors, ~100x the storageof a single vector). Query time is cheap: one document encode per query,not one per document. That asymmetry is the entire argument for reranking.**SciFact** — 5,183 scientific-claim documents, 300 judged queries. Claims areshort and self-contained. Published BEIR BM25 nDCG@10 sits near 0.66-0.69.

In [1]:
%pip install -q --upgrade 'transformers>=5.19.0' 'sentence-transformers>=6.0.0' datasets rank_bm25 pytrec_eval-terrier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 19.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
import torch, sys, transformers, sentence_transformers
print("python", sys.version.split()[0])
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("sentence-transformers", sentence_transformers.__version__)

# embedding_gemma2 landed in a recent transformers release. If this is older
# the Gemma run dies deep inside model loading with a confusing message.
from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
ok = "embedding_gemma2" in CONFIG_MAPPING_NAMES
print("embedding_gemma2 supported:", ok)
if not ok:
    raise RuntimeError(
        "transformers %s is too old for google/embeddinggemma-2. "
        "Run: %%pip install -q --upgrade 'transformers>=5.19.0'"
        % transformers.__version__
    )

print()
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"vram: {gb:.1f} GB")
    print()
    if not torch.cuda.is_bf16_supported():
        print("!! No native bf16 (Turing-class, e.g. T4).")
        print("!! EmbeddingGemma 2 MUST run fp32 -- fp16 returns NaN silently.")
else:
    print("!! No GPU. This notebook will be very slow or fail.")

python 3.13.15
torch 2.11.0+cu130
transformers 5.19.0
sentence-transformers 6.1.0
embedding_gemma2 supported: True

cuda available: True
device: Tesla T4
bf16 supported: True
vram: 14.6 GB



In [3]:
%%writefile beir_ab.py
"""
BEIR A/B harness: EmbeddingGemma 2 vs pplx-embed-v2-late-0.6b

Design notes
------------
pylate is NOT usable here: pylate 1.6.0 pins sentence-transformers==5.3.0 and
transformers<=5.3.0, while pplx-embed-v2-late requires >=6.0.0 / >=5.4.0.
So scoring is brute-force MaxSim, which is fine at BEIR scale
(SciFact 5.2k docs, NFCorpus 3.6k docs) and needs no index.

Two silent-failure traps are handled explicitly:
  * EmbeddingGemma 2 needs task instruction prefixes, and needs
    documents formatted as "title: {title} | text: {content}".
  * fp16 returns NaN / degraded embeddings. We use bf16 if the GPU
    supports it, else fp32. Never fp16.
"""

from __future__ import annotations

import argparse
import json
import re
import time
from pathlib import Path

import numpy as np

# --------------------------------------------------------------------- data

DATASETS = {
    "scifact":  {"corpus": 5183, "queries": 300},
    "nfcorpus": {"corpus": 3633, "queries": 323},
    "scidocs":  {"corpus": 25657, "queries": 1000},
}


def load_beir(name: str, cache_dir: str | None = None):
    """Load a BEIR dataset as (corpus_ids, corpus_texts, query_ids, queries, qrels).

    corpus_texts are already formatted for EmbeddingGemma 2 document input.
    """
    from datasets import load_dataset

    corpus = load_dataset(f"BeIR/{name}", "corpus", split="corpus",
                          cache_dir=cache_dir)
    queries = load_dataset(f"BeIR/{name}", "queries", split="queries",
                           cache_dir=cache_dir)
    # qrels live in the *_qrels dataset; fall back to the eval split
    qrels_ds = load_dataset(f"BeIR/{name}-qrels", split="test",
                            cache_dir=cache_dir)

    c_ids = [str(x) for x in corpus["_id"]]
    titles = corpus["title"] if "title" in corpus.column_names else [""] * len(c_ids)
    texts = corpus["text"]
    c_texts = [
        f"title: {(t or '').strip() or 'none'} | text: {x.strip()}"
        for t, x in zip(titles, texts)
    ]

    q_ids, q_texts = [], []
    for row in queries:
        text = (row.get("text") or "").strip()
        if not text:
            continue
        q_ids.append(str(row["_id"]))
        q_texts.append(text)

    qrels: dict[str, dict[str, int]] = {}
    for row in qrels_ds:
        qid, did, rel = str(row["query-id"]), str(row["corpus-id"]), int(row["score"])
        if rel > 0:
            qrels.setdefault(qid, {})[did] = rel

    # keep only queries that have at least one positive judgement
    keep = [i for i, qid in enumerate(q_ids) if qid in qrels]
    q_ids = [q_ids[i] for i in keep]
    q_texts = [q_texts[i] for i in keep]

    return c_ids, c_texts, q_ids, q_texts, qrels


# ------------------------------------------------------------------ metrics

def evaluate(run: dict[str, dict[str, float]], qrels: dict[str, dict[str, int]],
             k: int = 10) -> dict[str, float]:
    """nDCG@k / Recall@k / MRR@k via pytrec_eval, matching BEIR conventions."""
    import pytrec_eval

    ev = pytrec_eval.RelevanceEvaluator(
        {q: {d: int(r) for d, r in docs.items()} for q, docs in qrels.items()},
        {f"ndcg_cut.{k}", f"recall_{k}", f"map_cut.{k}", "recip_rank"},
    )
    per = ev.evaluate(run)

    out = {}
    for metric in (f"ndcg_cut_{k}", f"recall_{k}", f"map_cut_{k}", "recip_rank"):
        vals = [v[metric] for v in per.values() if metric in v]
        out[metric] = float(np.mean(vals)) if vals else 0.0
    return out


# ------------------------------------------------------------- retrievers

class BM25Retriever:
    """Lexical baseline. Needs no GPU; sanity-checks the eval path."""

    def __init__(self, k1: float = 0.9, b: float = 0.4):
        self.k1, self.b = k1, b

    def run(self, c_texts, q_texts, batch=64):
        from rank_bm25 import BM25Okapi
        tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
        bm = BM25Okapi([tok(t) for t in c_texts])
        return {qi: {di: float(s) for di, s in
                     zip(range(len(c_texts)), bm.get_scores(tok(q)))}
                for qi, q in enumerate(q_texts)}


class GemmaRetriever:
    """Dense single-vector. Correct prefixes, bf16-or-fp32, never fp16."""

    def __init__(self, model_id="google/embeddinggemma-2", dim=768, batch=16):
        import torch
        from sentence_transformers import SentenceTransformer
        self.torch = torch
        self.dim, self.batch = dim, batch
        self.model = SentenceTransformer(
            model_id,
            model_kwargs={"torch_dtype": (
                torch.bfloat16 if torch.cuda.is_available()
                and torch.cuda.is_bf16_supported() else torch.float32)},
            config_kwargs={"vision_config": None, "audio_config": None},
        )

    def encode(self, texts, is_query: bool):
        # Queries get the SearchQuery task prefix. Documents are already
        # formatted as "title: {title} | text: {content}" by load_beir,
        # which IS the document-side instruction for titled documents, so
        # they must NOT receive the query prefix. Getting this wrong
        # degrades quality silently rather than raising.
        kw = dict(batch_size=self.batch, truncate_dim=self.dim,
                  normalize_embeddings=True, show_progress_bar=True)
        if is_query:
            kw["prompt_name"] = "SearchQuery"
        return self.model.encode(texts, **kw).astype("float32")

    def run(self, c_texts, q_texts):
        c = self.encode(c_texts, False)
        q = self.encode(q_texts, True)
        sims = q @ c.T                      # (nq, ndoc)
        return {i: dict(enumerate(row)) for i, row in enumerate(sims)}


def _to_numpy(x) -> np.ndarray:
    if hasattr(x, "detach"):
        return x.detach().cpu().float().numpy()
    return np.asarray(x, dtype="float32")


class LateRetriever:
    """ColBERT-style late interaction, brute-force MaxSim, no index needed.

    128 dims per token; score = sum over query tokens of max over doc tokens.
    """

    def __init__(self, model_id="perplexity-ai/pplx-embed-v2-late-0.6b",
                 batch=8, doc_batch=64):
        from sentence_transformers import MultiVectorEncoder
        self.model = MultiVectorEncoder(model_id, device="cuda" if _cuda() else "cpu")
        self.batch, self.doc_batch = batch, doc_batch

    def run(self, c_texts, q_texts):
        import torch
        print(f"  encoding {len(q_texts)} queries...", flush=True)
        qv = self.model.encode_query(q_texts, batch_size=self.batch)
        # MultiVectorEncoder returns a list of (n_tokens, dim) CUDA tensors or arrays
        qv = [_to_numpy(x) for x in qv]

        scores = np.zeros((len(q_texts), len(c_texts)), dtype="float32")
        n_batches = (len(c_texts) + self.doc_batch - 1) // self.doc_batch
        print(f"  encoding {len(c_texts)} docs in {n_batches} batches...", flush=True)
        for b_idx, d0 in enumerate(range(0, len(c_texts), self.doc_batch)):
            d1 = min(d0 + self.doc_batch, len(c_texts))
            dv = [_to_numpy(x)
                  for x in self.model.encode_document(c_texts[d0:d1],
                                                      batch_size=self.batch)]
            for qi in range(len(q_texts)):
                scores[qi, d0:d1] = maxsim(qv[qi], dv)
            if (b_idx + 1) % 10 == 0 or b_idx == n_batches - 1:
                print(f"    processed docs {d1}/{len(c_texts)} (batch {b_idx+1}/{n_batches})", flush=True)
        return {i: dict(enumerate(row)) for i, row in enumerate(scores)}


def maxsim(q: np.ndarray, docs: list[np.ndarray]) -> np.ndarray:
    """sum_t max_d (q_t . d_d) for one query against many docs."""
    out = np.empty(len(docs), dtype="float32")
    for j, d in enumerate(docs):
        if q.size == 0 or d.size == 0:
            out[j] = -1e9
            continue
        out[j] = (d @ q.T).max(axis=0).sum()
    return out


def _cuda():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


# --------------------------------------------------------------------- main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", default="scifact", choices=list(DATASETS))
    ap.add_argument("--models", default="bm25,gemma,late")
    ap.add_argument("--dim", type=int, default=768)
    ap.add_argument("--k", type=int, default=10)
    ap.add_argument("--limit-queries", type=int, default=0)
    ap.add_argument("--out", default="results.json")
    args = ap.parse_args()

    print(f"loading {args.dataset} ...", flush=True)
    c_ids, c_texts, q_ids, q_texts, qrels = load_beir(args.dataset)
    if args.limit_queries:
        c_ids, c_texts = c_ids, c_texts
        q_ids, q_texts = q_ids[:args.limit_queries], q_texts[:args.limit_queries]
        qrels = {q: r for q, r in qrels.items() if q in set(q_ids)}
    print(f"  corpus={len(c_ids):,}  queries={len(q_ids)}  "
          f"judged queries={len(qrels)}", flush=True)

    results = {"dataset": args.dataset, "corpus": len(c_ids),
               "queries": len(q_ids),
               "judged": len(qrels), "k": args.k, "models": {}}

    for name in args.models.split(","):
        name = name.strip()
        print(f"\n=== {name} ===", flush=True)
        t0 = time.perf_counter()
        try:
            if name == "bm25":
                r = BM25Retriever()
            elif name == "gemma":
                r = GemmaRetriever(dim=args.dim)
            elif name == "late":
                r = LateRetriever()
            else:
                raise SystemExit(f"unknown model {name}")

            run = r.run(c_texts, q_texts)
            # map int keys back to real doc ids
            mapped = {q_ids[qi]: {c_ids[di]: float(s)
                                  for di, s in docs.items()}
                      for qi, docs in run.items() if q_ids[qi] in qrels}
            m = evaluate(mapped, qrels, k=args.k)
            m["seconds"] = round(time.perf_counter() - t0, 1)
            results["models"][name] = m
            print(f"  nDCG@{args.k}={m[f'ndcg_cut_{args.k}']:.4f}  "
                  f"R@{args.k}={m[f'recall_{args.k}']:.4f}  "
                  f"MRR={m['recip_rank']:.4f}  ({m['seconds']}s)", flush=True)
        except Exception as e:
            results["models"][name] = {"error": f"{type(e).__name__}: {e}"}
            print(f"  FAILED: {type(e).__name__}: {e}", flush=True)

    Path(args.out).write_text(json.dumps(results, indent=2))
    print(f"\nwrote {args.out}")


if __name__ == "__main__":
    main()

Writing beir_ab.py


## RunBM25 runs first as a control — it needs no GPU and finishes in seconds.

In [4]:
%run beir_ab.py --dataset scifact --models bm25,late --k 10 --out results_scifact.json

loading scifact ...


README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.47MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/5183 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]

  encoding 300 queries...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


  encoding 5183 docs in 81 batches...
    processed docs 640/5183 (batch 10/81)
    processed docs 1280/5183 (batch 20/81)
    processed docs 1920/5183 (batch 30/81)
    processed docs 2560/5183 (batch 40/81)
    processed docs 3200/5183 (batch 50/81)
    processed docs 3840/5183 (batch 60/81)
    processed docs 4480/5183 (batch 70/81)
    processed docs 5120/5183 (batch 80/81)
    processed docs 5183/5183 (batch 81/81)
  nDCG@10=0.7819  R@10=0.9016  MRR=0.7535  (719.1s)

wrote results_scifact.json


In [5]:
import json
r = json.load(open("results_scifact.json"))
rows = [("model", "nDCG@10", "Recall@10", "MRR", "secs")]
for name, m in r["models"].items():
    if "error" in m:
        rows.append((name, "ERROR", m["error"][:44], "", ""))
    else:
        rows.append((name, f"{m['ndcg_cut_10']:.4f}",
                     f"{m['recall_10']:.4f}", f"{m['recip_rank']:.4f}",
                     str(m["seconds"])))
w = [max(len(x[i]) for x in rows) for i in range(5)]
for row in rows:
    print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(row)))
print()
print(f"corpus={r['corpus']}  queries={r['queries']}  judged={r['judged']}")

model  nDCG@10  Recall@10  MRR     secs 
bm25   0.6532   0.7790     0.6234  17.1 
late   0.7819   0.9016     0.7535  719.1

corpus=5183  queries=300  judged=300


## Reading the output`ndcg_cut_10` is the headline BEIR number, identical to what papers report.`recall_10`, `map_cut_10` and `recip_rank` (MRR) come along for free.Two things to check before trusting a run:1. **BM25 should land in a known range.** If it does not, the data or eval   path is broken and every model number in this run is suspect. Reference   nDCG@10 at `k1=0.9, b=0.4` on BEIR's own splits: SciFact ~0.66,   NFCorpus ~0.32. NFCorpus is *supposed* to be far lower — it uses graded   relevance and has many relevant documents per query.Whichever model wins here answers only one question: *does a rerank passearn its cost on short documents?* Both datasets here are short andself-contained, which favours single-vector dense retrieval. That is not theregime your 250M-token book library lives in.